# 49. The direct probe on a different dataset

## Why this exists

Notebook 47 measured how well each model fits a task it has never seen, at the
start of the stream and at the end. Every run fitted the new task better at the
end. That is the paper's central measurement.

It has one weakness, and a reviewer will find it. The probe task is group 9 of
Tiny ImageNet, and the stream learns groups 0 to 8 of the same dataset. So the
model that has learned nine groups may be better at group 9 simply because the
pictures are alike. That is ordinary transfer between related tasks, not a
statement about the ability to learn.

This notebook repeats the same measurement with a probe task that shares
nothing with the stream.

## Why CIFAR-100 and not EuroSAT

EuroSAT would be the natural choice: satellite photographs, already used in
this study as images for the indicator readings, already 64 by 64.

It does not fit. The generator's output layer is built once, with as many
outputs as a stream task has classes, which is twenty. A probe task must have
exactly twenty classes. EuroSAT has ten. Under the old ten-class design it
matched, and `uncle/eurosat_probe/` in R2 holds those runs. The twenty-class
split ended that.

CIFAR-100 has one hundred classes, so twenty of them can be taken as one task
with the right shape. The pictures are 32 by 32 and are enlarged to 64 by 64,
the size the backbone expects.

## What this can and cannot settle

It can settle whether the gain in Section 5.2 survives when the probe task
shares no classes and no source with the stream.

It cannot remove transfer altogether. A model trained on natural images holds
general visual features, and those help on any natural-image task. Against a
randomly started model at boundary 0, that help is part of what is being
measured. The honest claim after this run is narrower than "no transfer": it is
"not transfer from neighbouring classes".

## What stays the same as notebook 47

Everything that would otherwise make the two incomparable. The same fifteen
streams, the same saved models, the same three learning rates with the best
chosen on a validation split, the same 250 steps, the same eleven repetitions,
the same 4,800 training images, the same fork-and-discard. Only the probe task
changes.

## What this costs

About half a minute per probe run. Order A alone is 330 runs, near three hours.
All fifteen streams is 990 runs, near eight hours. The settings cell starts
with order A only, on purpose: look at the answer before paying for the rest.
Finished runs are saved to R2 as they go and skipped on a restart, so the work
can be split across sessions.


## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-probe")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition_10x20.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this from a checkout that has the ten-group split.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None and IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

import torch
from tqdm.auto import tqdm
from uncle import checkpoint
from uncle.config import Config, PROBE_TASK, study_defaults
from uncle.hypernet import HyperNetwork, build_target
from uncle.plasticity_indicators import indicator_key
from uncle.probe import noise_floor, probe_once, snapshot, unchanged
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS
from uncle.trainer import UnCLe

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| device:", GPU_NAME)
if DEVICE == "cpu":
    print("No GPU. This notebook trains hundreds of small forks; it needs one.")

## 2. Settings

In [ ]:
STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

# Start with order A only. Five streams is 330 probe runs, near three hours,
# and that is enough to see whether the gain survives a different dataset.
#
# If it does, add the other two orderings here and run again. Nothing already
# finished is repeated, because every run is matched by identity against the
# saved file before it is started.
ORDERS = ["order_A"]
SEEDS = [0, 1, 2, 3, 4]

# Three points. 0 is the untrained generator, which the seed rebuilds exactly
# and which needs no saved file. 9 is the end of the nine task stream. 4 is the
# middle, and it is here because notebook 48 found that on the Tiny ImageNet
# probe the whole gain is finished by task 4 and flat afterwards. Whether the
# CIFAR gain behaves the same way is the live question.
#
# Adding 4 does not repeat anything. A finished run is matched by identity, and
# the identity includes the point, so a second run of this notebook performs
# only the 165 runs at boundary 4.
POINTS = [0, 4, 9]

# A new label, because the probe task changed. Runs are matched against the
# saved file by identity, and a shared label would quietly mix two protocols
# into one file.
RUN_LABEL = "cifar100_20class_train4800"

RESUME_KEY = f"uncle/matched_probe/{STUDY_ID}/{RUN_LABEL}.jsonl"
SAVE_EVERY = 10

# The probe task name. Not a digit and not a Tiny ImageNet group, so it can
# never collide with a stream task.
PROBE_TASKS = ["cifar100"]
CLASSES_HINT = 20

# Which twenty of CIFAR-100's hundred classes. Fixed here rather than drawn at
# run time, so a rerun on another machine reads the same task. The choice is
# arbitrary; what matters is that it never changes.
CIFAR_CLASS_SEED = 7
CIFAR_CLASSES_WANTED = 20

# Identical to notebook 47. Changing any of these would make the two probes
# incomparable, which is the whole point of running this one.
RATES = [1e-4, 3e-4, 1e-3]
STEPS = 250
BATCH_SIZE = 64
FEATURE_IMAGES = 2000
REPETITIONS_AT_ZERO = 11
REPETITIONS_LATER = 11

# (name, generator frozen, code trains). "full" trains the generator together
# with a fresh task code, which is what notebook 47 ran.
ARMS = [("full", False, True)]

# Per class. Twenty classes at 240 images is 4,800 training images, the same
# total notebook 47 used, so the model sees the same number of images at both
# probes. CIFAR-100 has 500 training images per class, so 240 and 20 fit with
# room to spare.
TRAIN_PER_CLASS = 240
VAL_PER_CLASS = 20
SPLIT_SEED = 606

DATA_ROOT = (Path("/content/data/tiny-imagenet-200") if IN_COLAB
             else REPO / "data/tiny-imagenet-200")
CIFAR_ROOT = Path("/content/data/cifar100") if IN_COLAB else REPO / "data/cifar100"
CACHE = Path("/content/probe-cifar") if IN_COLAB else REPO / "outputs/probe-cifar100"
CACHE.mkdir(parents=True, exist_ok=True)

RUNS = [(order, seed) for order in ORDERS for seed in SEEDS]
TOTAL = len(RUNS) * len(POINTS) * len(RATES) * REPETITIONS_AT_ZERO
print(f"{len(RUNS)} streams x {len(POINTS)} points x {len(RATES)} rates x "
      f"{REPETITIONS_AT_ZERO} repetitions = {TOTAL} probe runs")
print(f"About {TOTAL * 0.5 / 60:.1f} hours at half a minute each.")

## 3. Connect to R2

In [ ]:
def secret(name):
    """Read a credential from Colab secrets, or the environment elsewhere."""
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    value = os.environ.get(name)
    if value is None:
        env = REPO / ".env"
        if env.exists():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith(name + "="):
                    return line.partition("=")[2].strip().strip('"').strip("'")
    return value


store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))

# The settings the streams were trained with, read back rather than retyped, so
# the model this notebook rebuilds has the shape its checkpoint expects.
plan = json.loads(store.download(
    f"uncle/indicators/{STUDY_ID}/job2_summary.json", CACHE).read_text(encoding="utf-8"))
CLASSES = plan["classes_per_task"]
assert CLASSES == CLASSES_HINT, f"settings assume {CLASSES_HINT}, the runs used {CLASSES}"
print("Streams used learning rate", plan["learning_rate"], "| epochs", plan["epochs"],
      "| batch", plan["batch_size"], "| classes per task", CLASSES)

# Stop now if a stream this notebook is about to probe has no end-of-stream
# model, rather than an hour into the session.
FORGET = {"order_A": "0", "order_B": "4", "order_C": "8"}
for order, seed in RUNS:
    for point in POINTS:
        if point == 0:
            continue
        key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
        if store.info(key) is None:
            raise RuntimeError(f"Missing {key}")
print("Every model this session needs is in R2.")

## 4. Build the CIFAR-100 probe task

In [ ]:
from torch.utils.data import Dataset, Subset
from torchvision.datasets import CIFAR100
from torchvision import transforms

# Tiny ImageNet is loaded only because the trainer is built around a task
# dictionary. The probe never reads these images: it is handed its own.
STREAM_TASKS = [str(i) for i in range(9)]
tasks = build_tasks(root=DATA_ROOT, partition_path=PARTITIONS[20],
                    partition_seed=42, include=tuple(STREAM_TASKS), download=True)
print("Loaded Tiny ImageNet images for", len(STREAM_TASKS), "stream tasks")

if set(PROBE_TASKS) & set(STREAM_TASKS):
    raise ValueError("The probe task name collides with a stream task.")

# ToTensor only, matching what the stream tasks use, so the probe images have
# the same scale as everything else this model has seen. The resize comes
# first, on the picture, which is cleaner than stretching a tensor.
to_model_size = transforms.Compose([transforms.Resize(64), transforms.ToTensor()])

cifar_train = CIFAR100(root=str(CIFAR_ROOT), train=True,
                       transform=to_model_size, download=True)
cifar_test = CIFAR100(root=str(CIFAR_ROOT), train=False,
                      transform=to_model_size, download=True)

# Twenty of the hundred classes, drawn once from a fixed seed and sorted, so
# the same twenty come out on any machine.
chooser = torch.Generator().manual_seed(CIFAR_CLASS_SEED)
chosen = sorted(torch.randperm(100, generator=chooser)[:CIFAR_CLASSES_WANTED].tolist())
# The model's output layer has twenty slots numbered 0 to 19, so the original
# CIFAR labels have to be renumbered into that range.
relabel = {old: new for new, old in enumerate(chosen)}
print("CIFAR-100 classes used:", chosen)
print("Names:", [cifar_train.classes[c] for c in chosen])


class TwentyOf(Dataset):
    """The rows of one CIFAR-100 split whose class is one of the chosen twenty.

    Keeps a `targets` list so the splitting code below can work on labels
    alone, without reading a single picture.
    """

    def __init__(self, base, relabel):
        self.base = base
        self.rows = [row for row, label in enumerate(base.targets) if label in relabel]
        self.targets = [relabel[base.targets[row]] for row in self.rows]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        image, _ = self.base[self.rows[index]]
        return image, self.targets[index]


cifar_task_train = TwentyOf(cifar_train, relabel)
cifar_task_test = TwentyOf(cifar_test, relabel)

# Three things that would quietly ruin the comparison if they were wrong.
assert len(set(cifar_task_train.targets)) == CLASSES, "Not twenty classes."
assert min(cifar_task_train.targets) == 0 and max(cifar_task_train.targets) == CLASSES - 1, \
    "Labels are not renumbered into the model's output range."
assert cifar_task_train[0][0].shape == (3, 64, 64), \
    f"Wrong picture size: {tuple(cifar_task_train[0][0].shape)}"


def balanced_split(labels, counts, seed):
    """Row numbers for disjoint splits, equal numbers from every class.

    Works on the labels alone, so no image is read here. The same function
    notebook 47 uses, so the two probes split their data the same way.
    """
    by_class = {}
    for row, label in enumerate(labels):
        by_class.setdefault(int(label), []).append(row)
    if len(by_class) != CLASSES:
        raise RuntimeError(f"Expected {CLASSES} classes, found {len(by_class)}.")
    stream = torch.Generator().manual_seed(seed)
    splits = [[] for _ in counts]
    for label, rows in sorted(by_class.items()):
        if len(rows) < sum(counts):
            raise RuntimeError(f"Class {label} has {len(rows)} images, needs {sum(counts)}.")
        order = [rows[j] for j in torch.randperm(len(rows), generator=stream).tolist()]
        at = 0
        for which, how_many in enumerate(counts):
            splits[which] += order[at:at + how_many]
            at += how_many
    taken = [row for split in splits for row in split]
    assert len(set(taken)) == len(taken), "An image landed in two splits."
    return splits


# Training and validation both come out of CIFAR-100's training half, so no
# picture is ever both trained on and used to pick a learning rate. The test
# half is left untouched for scoring.
train_rows, val_rows = balanced_split(cifar_task_train.targets,
                                      (TRAIN_PER_CLASS, VAL_PER_CLASS), SPLIT_SEED)
probe_sets = {PROBE_TASKS[0]: {"train": Subset(cifar_task_train, train_rows),
                               "val": Subset(cifar_task_train, val_rows),
                               "test": cifar_task_test}}

for name, split in probe_sets.items():
    print(f"task {name}: train {len(split['train'])}  val {len(split['val'])}  "
          f"test {len(split['test'])}")
print(f"Chance on this task is {100 / CLASSES:.1f} percent, the same as notebook 47.")

## 5. Building a model at one probe point

In [ ]:
def build(order, seed, point):
    """The model this stream had at `point`. Point 0 needs no file."""
    settings = study_defaults(order)
    settings["requests"] = settings["requests"] + (("forget", FORGET[order]),)
    config = Config(**settings, seed=seed,
                    learning_rate=plan["learning_rate"], epochs=plan["epochs"],
                    batch_size=plan["batch_size"],
                    initialization=plan["initialization"], device=DEVICE)
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    if point > 0:
        key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
        payload = torch.load(store.download(key, CACHE), map_location=DEVICE,
                             weights_only=False)
        checkpoint.restore(payload, hypernet=hypernet, uncle=trainer)
    return target, hypernet, trainer

## 6. Run the grid

In [ ]:
def identity(run):
    """Everything that makes one probe run different from another."""
    return (run["order"], run["seed_label"], run["point"], run["task"],
            run["learning_rate"], run["arm"], run["repetition"])


def probe_seed(task, repetition):
    """Fixes the batch order and the fresh code.

    Depends on the task and the repetition, never on the probe point, so two
    points differ only by the state of the model being probed. The task name
    here is a word rather than a number, so it falls to the same constant
    notebook 47 uses for a non-numeric name.
    """
    return 12345 + 1000 * repetition + (int(task) if task.isdigit() else 99)


PARTIAL = CACHE / "probe_partial.jsonl"
results = []
if store.info(RESUME_KEY) is not None:
    saved = store.download(RESUME_KEY, CACHE)
    for line in saved.read_text(encoding="utf-8").splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            print("Skipped one unreadable line at the end of the saved file.")
    print(f"Resuming: {len(results)} runs already finished.")
else:
    print("Starting fresh: nothing saved in R2 yet.")

PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in results), encoding="utf-8")
done = {identity(r) for r in results}
new_runs = 0


def grid_at(order, seed, point, repetitions):
    """Every rate, arm and repetition at one point of one stream."""
    global new_runs
    todo = [(task, rate, arm, code_only, train_code, rep)
            for task in PROBE_TASKS
            for rate in RATES
            for arm, code_only, train_code in ARMS
            for rep in range(repetitions)
            if (order, seed, point, task, rate, arm, rep) not in done]
    if not todo:
        print(f"{order} seed {seed}, t = {point}: already finished")
        return

    target, hypernet, trainer = build(order, seed, point)
    reference = snapshot(hypernet, trainer)
    bar = tqdm(todo, desc=f"{order} seed {seed}  t={point}", unit="run")
    for task, rate, arm, code_only, train_code, rep in bar:
        bar.set_postfix_str(f"lr={rate:g} rep {rep + 1}")
        started = time.perf_counter()
        run = probe_once(
            hypernet, target, trainer, task=task,
            train_set=probe_sets[task]["train"], test_set=probe_sets[task]["test"],
            validation_set=probe_sets[task]["val"],
            steps=STEPS, learning_rate=rate, batch_size=BATCH_SIZE,
            code_only=code_only, train_code=train_code,
            seed=probe_seed(task, rep), feature_images=FEATURE_IMAGES)
        # A probe forks and discards. If it changed the model, every later
        # reading in this session is measuring something that has moved.
        if not unchanged(hypernet, reference):
            raise RuntimeError("A probe changed the model. Stop and fix this "
                               "before trusting anything.")
        run.update(order=order, point=point, endpoint="E2", repetition=rep,
                   seed_label=seed, minutes=(time.perf_counter() - started) / 60,
                   device=GPU_NAME, run_label=RUN_LABEL, code_revision=HEAD,
                   train_images=len(probe_sets[task]["train"]),
                   probe_dataset="cifar100", probe_classes=chosen)
        results.append(run)
        done.add(identity(run))
        with PARTIAL.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(run) + "\n")
        new_runs += 1
        if new_runs % SAVE_EVERY == 0:
            store.upload(PARTIAL, RESUME_KEY, overwrite=True)
    bar.close()

    del trainer, hypernet, target
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    store.upload(PARTIAL, RESUME_KEY, overwrite=True)


began = time.perf_counter()
for order, seed in RUNS:
    for point in POINTS:
        grid_at(order, seed, point,
                REPETITIONS_AT_ZERO if point == 0 else REPETITIONS_LATER)
print(f"\nFinished in {(time.perf_counter() - began) / 3600:.1f} hours. "
      f"{len(results)} runs in total.")

## 7. The result, beside the Tiny ImageNet probe

In [ ]:
import math
import statistics
from collections import defaultdict


def best_per_repetition(rows):
    """Test accuracy per (order, seed, point), using the study's selection rule.

    The learning rate is chosen inside each repetition, on that repetition's
    own validation accuracy, with ties going to the lower rate. Choosing on the
    mean instead gives a different number, so this matches notebook 47 exactly.
    """
    cells = defaultdict(list)
    for row in rows:
        cells[(row["order"], row["seed_label"], row["point"],
               row["repetition"])].append(row)
    picked = defaultdict(list)
    for key, arms in cells.items():
        best = min(arms, key=lambda a: (-a["val_accuracy"], a["learning_rate"]))
        picked[key[:3]].append(best["test_accuracy"])
    return picked


def summarize(rows, label):
    """Start, end, gain and noise floor for every stream in `rows`."""
    picked = best_per_repetition(rows)
    streams = sorted({key[:2] for key in picked})
    print()
    print(label)
    print(f"{'run':6s} {'start':>7s} {'end':>7s} {'gain':>7s} {'floor':>6s}  clears")
    gains = []
    for order, seed in streams:
        at0, at9 = picked[(order, seed, 0)], picked[(order, seed, 9)]
        if len(at0) < 2 or len(at9) < 2:
            continue
        gain = statistics.mean(at9) - statistics.mean(at0)
        # The smallest change that would survive the repetition spread at the
        # 5 percent level, built from the boundary 0 repetitions. Same formula
        # as the paper's Section 5.2.
        floor = 1.96 * statistics.stdev(at0) * math.sqrt(1 / len(at0) + 1 / len(at9))
        gains.append(gain)
        print(f"{order[-1]}{seed:<5d} {statistics.mean(at0):7.2f} "
              f"{statistics.mean(at9):7.2f} {gain:+7.2f} {floor:6.2f}  "
              f"{'yes' if gain > floor else 'NO'}")
    if gains:
        print(f"mean gain {statistics.mean(gains):+.2f} points over "
              f"{len(gains)} streams, chance {100 / CLASSES:.1f} percent")

    # When boundary 4 is present, say where in the stream the gain happened.
    # A gain that is finished by task 4 and flat afterwards means the stream
    # stops changing early, whatever the probe task is.
    middle = [(order, seed) for order, seed in streams
              if len(picked.get((order, seed, 4), [])) >= 2]
    if middle:
        print()
        print(f"{'run':6s} {'t=0':>7s} {'t=4':>7s} {'t=9':>7s} "
              f"{'0 to 4':>8s} {'4 to 9':>8s}")
        early, late = [], []
        for order, seed in middle:
            at0 = statistics.mean(picked[(order, seed, 0)])
            at4 = statistics.mean(picked[(order, seed, 4)])
            at9 = statistics.mean(picked[(order, seed, 9)])
            early.append(at4 - at0)
            late.append(at9 - at4)
            print(f"{order[-1]}{seed:<5d} {at0:7.2f} {at4:7.2f} {at9:7.2f} "
                  f"{at4 - at0:+8.2f} {at9 - at4:+8.2f}")
        print(f"mean 0 to 4 {statistics.mean(early):+.2f} points, "
              f"mean 4 to 9 {statistics.mean(late):+.2f} points")
        if len(late) > 1:
            # The standard error says whether the late change is separable
            # from no change at all, which is what notebook 48 asked.
            spread = statistics.stdev(late) / len(late) ** 0.5
            print(f"the 4 to 9 change is {abs(statistics.mean(late)) / spread:.1f} "
                  f"standard errors from zero on {len(late) - 1} degrees of freedom")
    return gains


cifar_gains = summarize(results, f"CIFAR-100 probe ({RUN_LABEL})")

# The Tiny ImageNet probe, read back from R2 for the same streams, so the two
# sit side by side rather than in two different places.
tiny_key = f"uncle/matched_probe/{STUDY_ID}/revised_20class_train4800.jsonl"
tiny_rows = [json.loads(line) for line in
             store.download(tiny_key, CACHE).read_text(encoding="utf-8").splitlines()
             if line.strip()]
tiny_rows = [r for r in tiny_rows if (r["order"], r["seed_label"]) in set(RUNS)]
tiny_gains = summarize(tiny_rows, "Tiny ImageNet probe (notebook 47), same streams")

if cifar_gains and tiny_gains:
    print()
    print(f"Same-dataset gain:      {statistics.mean(tiny_gains):+.2f} points")
    print(f"Different-dataset gain: {statistics.mean(cifar_gains):+.2f} points")
    share = 100 * statistics.mean(cifar_gains) / statistics.mean(tiny_gains)
    print(f"The different-dataset probe keeps {share:.0f} percent of the gain.")

## 8. What this decides

Read the last three lines of the cell above.

**If the CIFAR-100 gain is still clearly positive and clears every floor**, then
the Section 5.2 result is not transfer from neighbouring classes. The model that
has learned nine Tiny ImageNet groups fits an unrelated twenty-class task better
than a randomly started model does. Section 7 can then narrow its transfer
caveat, and the other two orderings are worth the five hours.

**If the gain shrinks a lot but stays positive**, the honest reading is that part
of Section 5.2's eighteen points was transfer and part was not. Both numbers
belong in the paper, and the claim becomes smaller and better supported.

**If the gain disappears or reverses**, then Section 5.2 measured transfer, and
the paper's central disagreement does not hold in the form it is written. That
is worth knowing before submission rather than after review.

In all three cases the numbers go into the trajectory record for the day, and
the paper is updated from the record, not from this output.

## What this still does not answer

The probe is read at boundary 0 and boundary 9 only. Section 5.1 shows the
indicators move almost entirely at the first task, so a reviewer can ask whether
the comparison is really untrained against trained rather than early against
late. Boundary 4 is saved for every stream. Adding `4` to `POINTS` answers that,
and costs half as much again as this run.
